In [47]:
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import plotly.express as px
import matplotlib.pyplot as plt
import pytrendseries
import math
 
 
import warnings
warnings.filterwarnings('ignore')
 
import os
import eikon as ek
import refinitiv.data as rd
 
 
#Scroll through DF
pd.set_option("display.max_rows", None, "display.max_columns", None)
 
 
#Defining Proxy
os.environ['NO_PROXY'] = 'localhost'
os.environ['NO_PROXY'] = '127.0.0.1'
 
 
#Open Session
rd.open_session()
 
#API Key
ek.set_app_key('b7f9e07dd1664cb2b48043ec3321b0e219bc776b')
 
#Todays Datetime
dt_now = datetime.datetime.now()

In [48]:
trb_pairs = [
    "EUR=TRB", 
    "EURCHF=TRB", 
    "EURJPY=TRB",
    "EURGBP=TRB", 
    "EURCZK=TRB", 
    "EURHUF=TRB" 
]

fix_pairs = [
    "EURUSDFIXMP=WM", 
    "EURCHFFIXMP=WM", 
    "EURJPYFIXMP=WM",
    "EURGBPFIXMP=WM", 
    "EURCZKFIXMP=WM", 
    "EURHUFFIXMP=WM" 
]

In [58]:
# 12 Uhr Fixing
df_12 = rd.get_history(
    universe=fix_pairs,
    fields=["MID_PRICE"],
    interval="1D",
    start="2026-05-13T11:59:00",
    end="2026-08-13T12:05:00",
    use_field_names_in_headers=True
)

# 13 Uhr Fixing
df_13 = rd.get_history(
    universe=fix_pairs,
    fields=["MID_PRICE"],
    interval="1D",
    start="2026-05-13T12:59:00",
    end="2026-08-13T13:05:00",
    use_field_names_in_headers=True
)

In [59]:
df_12.index = pd.to_datetime(df_12.index) + pd.Timedelta(hours=12)
df_13.index = pd.to_datetime(df_13.index) + pd.Timedelta(hours=13)

result = (
    pd.concat([df_12, df_13])
      .sort_index()
)

print(result)

MID_PRICE            EURUSDFIXMP=WM  EURCHFFIXMP=WM  EURJPYFIXMP=WM  EURGBPFIXMP=WM  EURCZKFIXMP=WM  EURHUFFIXMP=WM
Date                                                                                                               
2026-05-14 12:00:00         1.16800         0.91460       184.71335         0.86615        24.31100       357.27500
2026-05-14 13:00:00         1.16800         0.91460       184.71335         0.86615        24.31100       357.27500
2026-05-15 12:00:00         1.16300         0.91455       184.44600         0.87145        24.32150       360.76500
2026-05-15 13:00:00         1.16300         0.91455       184.44600         0.87145        24.32150       360.76500
2026-05-18 12:00:00         1.16420         0.91495       184.92735         0.86915        24.31200       360.85000
2026-05-18 13:00:00         1.16420         0.91495       184.92735         0.86915        24.31200       360.85000
2026-05-19 12:00:00         1.15965         0.91665       184.45975     

In [54]:
import pandas as pd

# gewünschte Uhrzeiten
target_hours = [12, 13]

result = []

for day in df_fx.index.normalize().unique():
    for hour in target_hours:
        target_time = day + pd.Timedelta(hours=hour)

        # erster Wert nach dem Zielzeitpunkt
        mask = df_fx.index >= target_time

        if mask.any():
            result.append(df_fx.loc[mask].iloc[0])

result = pd.DataFrame(result)

AttributeError: 'RangeIndex' object has no attribute 'normalize'

In [50]:
print(df_fx)

MID_PRICE   EURUSDFIXMP=WM  EURCHFFIXMP=WM  EURJPYFIXMP=WM  EURGBPFIXMP=WM  EURCZKFIXMP=WM  EURHUFFIXMP=WM
Date                                                                                                      
2026-05-14         1.16800         0.91460       184.71335         0.86615        24.31100       357.27500
2026-05-15         1.16300         0.91455       184.44600         0.87145        24.32150       360.76500


In [51]:
print(df_fx.index)


DatetimeIndex(['2026-05-14', '2026-05-15'], dtype='datetime64[us]', name='Date', freq=None)


In [ ]:
import pandas as pd
import time
import refinitiv.data as rd

start_date = "2026-05-13"
end_date = "2026-08-13"

dates = pd.bdate_range(
    start=start_date,
    end=end_date
)

all_data = []
failed_dates = []

for number, date in enumerate(dates, start=1):
    date_string = date.strftime("%Y-%m-%d")

    try:
        df_day = rd.get_history(
            universe=trb_pairs,
            fields=["MID_PRICE"],
            interval="60min",
            start=f"{date_string}T12:00:00",
            end=f"{date_string}T13:01:00",
            use_field_names_in_headers=True
        )

        if df_day is not None and not df_day.empty:
            df_day.index = pd.to_datetime(df_day.index)

            # Nur exakt 12:00 und 13:00 behalten
            df_day = df_day[
                (df_day.index.strftime("%H:%M") == "12:00") |
                (df_day.index.strftime("%H:%M") == "13:00")
            ]

            if not df_day.empty:
                all_data.append(df_day)

            print(
                f"[{number}/{len(dates)}] "
                f"{date_string}: {len(df_day)} Zeitpunkte"
            )
        else:
            print(
                f"[{number}/{len(dates)}] "
                f"{date_string}: keine Daten"
            )

        time.sleep(0.2)

    except Exception as error:
        failed_dates.append(date_string)

        print(
            f"[{number}/{len(dates)}] "
            f"{date_string}: Fehler: {error}"
        )

        time.sleep(1)

# Alle Tage zusammenführen
if all_data:
    df_fx = pd.concat(all_data).sort_index()

    df_fx = df_fx[
        ~df_fx.index.duplicated(keep="last")
    ]
else:
    df_fx = pd.DataFrame()

print("\nDownload abgeschlossen")
print(f"Anzahl Zeilen: {len(df_fx)}")
print(f"Fehlerhafte Tage: {failed_dates}")

display(df_fx.head())

In [ ]:
df_fx.to_csv(
    "fx_mid_prices_12_13.csv",
    index=True,
    index_label="Timestamp",
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

print("CSV erfolgreich gespeichert.")